# Трансформеры. Encoder. Decoder. Механизм внимания (Attention)

## Where were we?

![alt](../data/bert.png)

В прошлый раз мы начали обсуждать обработку естественного языка (`NLP`). Поговорили о следующих этапах:

1. Токенизация - разбиение исходной последовательности на атомарные части (токены)
2. Предобработка текста
3. Векторизация токенов. Мы обсуждали способ получения `embedding` представления
4. Поговорили о том, что `RNN` эффективно может работать с последовательностями

![alt](../data/RNN.png)


Напомним, чтобы хранить информацию о предыдущих токенах, мы вводили понятие скрытого состояния (`hidden state`, векторы $h_n$). По сути, это некоторый вектор фиксированной размерности. На каждом шаге в RNN подаются токены (на самом деле мы работаем с тензором вида: `[token, batch, embedding_size]`), при этом происходит обновление скрытого состояния. Наше пространство объектов $X$ - это набор embeddings исходных токенов. Пусть $x_n$ - очередной вход, тогда:

$$h_n = tanh(h_{n-1}W_1 + x_nW_2)$$

Использование `tanh` в качестве функции активации позволяет нам бороться с двумя извечными проблемами `RNN`: затухающим градиентом и взрывом градиента.

`RNN` можно обучать на ошибку, равную суммарному отклонению по всем выходным $y_n$ нашей сети.


А ещё `RNN` может быть глубокой. Почему это так? Мы знаем, что `RNN` работает с последовательностью (например, с последовательностью токенов), однако `hidden states` $h_i$ тоже образую последовательность, которую можно передать на вход следующему рекуррентному слою `RNN`. Здесь аналогия такая же, как и с `convolution layers` в `CNN`:

![alt](../data/deepRNN.png)



На прошлом семинаре обсуждались архитектуры `RNN` такие как `LSTM` и `GRU`, показавшие себя наилучшим образом при работе с последовательностями. Однако стоит сказать, что стандартная `RNN` учитывает только предыдущий контекст. Но ведь слово в предложении связано не только с предыдущими, но и с последующими словами. В таких случаях имеет смысл использовать двунаправленную рекуррентную сеть (`bidirectional RNN`, `BRNN`). Как следует из названия, в bidirectional RNN есть две рекуррентных подсети: прямая (forward, токены в нее подаются от первого к последнему) и обратная (backward, токены подаются в обраттном порядке).

Вот пример такой архитектуры:

![alt](../data/BRNN.svg)


Обратите внимание, что двунаправленная рекуррентная сеть работает с входом фиксированного размера, и по-прежнему не может решать не синхронизованный вариант задачи many-to-many. Backward RNN должна точно знать, где заканчивается входная последовательность, чтобы начать её обрабатывать с конца. Зато такая архитектура может помочь в решении задачи определения именованных сущностей или частей речи, использоваться в качестве энкодера в машинном переводе и так далее.

## Seq2Seq

Вы, должно быть обратили внимание, что мы пока не касались задач, связанных с порождением последовательностей (синхронизованный варианты many-to-many не в счёт).

Действительно: имевшиеся у нас пока инструменты не позволяли генерировать последовательности произвольной длины. Но как тогда переводить с одного языка на другой? Ведь мы не знаем, какой должна быть длина перевода фразы, да и однозначного соответствия между словами исходного предложения и его перевода обычно нет.

Введем понятие архитектуры `encoder-decoder`:

Архитектура `encoder-decoder` (кодировщик-декодировщик), решает именно эту задачу. Она состоит из двух основных частей:

1. `Encoder` (кодировщик) — принимает на вход исходную последовательность токенов переменной длины и сжимает всю информацию о ней в некоторый вектор контекста (`context vector`), который часто называют скрытым состоянием (`hidden state`). Обычно это последнее скрытое состояние `RNN` (`LSTM` или `GRU`).

2. `Decoder` (декодировщик) — принимает этот вектор контекста и начинает пошагово порождать выходную последовательность, по одному токену за раз. На каждом шаге `decoder` предсказывает следующий токен, используя предыдущий предсказанный токен как вход.

![alt](../data/encoder-decoder.png)

Кажется, что мы уже с вами наблюдали похожую архитектуру, когда говорили про Word2Vec, но это немного другое..


Очевидным выбором на роль энкодера и декодера являются рекуррентные сети, например, LSTM. Простейшая архитектура будет иметь вид:

![alt](../data/lstm.gif)


1. `Encoder` читает входное предложение токен за токеном и обрабатывает их с помощью блоков рекуррентной сети. `Hidden state` последнего блока становится контекстным вектором. Часто энкодер читает предложение в обратном порядке. Это делается для того, чтобы последний токен, который видит `encoder`, совпал (или примерно совпал) с первыми токенами, которые будет генерировать `decoder`. Таким образом, декодеру проще начать процесс воссоздания предложения. Несколько первых правильных токенов сильно упрощают процесс дальнейшей генерации.

2. Архитектура `decoder` аналогична `encoder`. При этом каждый блок `decoder` должен учитывать токены, сгенерированные к текущему моменту, и также информацию о предложении на исходном языке. Вектор скрытого состояния в нулевом блоке `decoder` $g_0$ инициализируется при помощи контекстного вектора (`context vector`).


## Attention

<video width="100%" controls>
  <source src="../data/attention.mp4" type="video/mp4">
  Ваш браузер не поддерживает видео.
</video>

Внимательно посмотрим на seq2seq модель для машинного перевода. Вся информация о предложении на исходном языке заключена в контекстном векторе, но разные слова в предложении могут иметь разную смысловую значимость и следовательно, должны учитываться с разными весами. Кроме того, при генерации разных частей перевода следует обращать внимание на разные части исходного предложения. Например, первое слово переведенной фразы нередко связано с первыми словами в предложении, поданном на вход энкодеру, а порой одно слово перевода передаёт смысл нескольких слов, разбросанных по исходному предложению


Механизм внимания (`attention`) реализует эту интуицию путем предоставления декодеру информации обо всех токенах исходного предложения на каждом шаге генерации. Рассмотрим классическую модель внимания, предложенную [Bahdanau et al](https://arxiv.org/abs/1409.0473). в 2014 году.

![alt](../data/attention.jpg)


Обозначим скрытые состояния (`hidden states`) энкодера $(h_0, h_1, ..., h_n)$, а скрытые состояния декодера $(s_0, s_1, ..., s_m)$. Отметим, что $h_n = s_0$ - это контекстный вектор (`context vector`).

* На каждом шаге декодера будем считать `attention scores`, умножая $s_i$ на вектор скрытого состояния каждого блока энкодера $(h_0, h_1, ..., h_n)$

Таким образом, получаем $n$ значений, указывающих, насколько каждый из токенов с номерами от $0$ до $n$ из исходной последовательности важен для генерации токена $i$:

$$e_i = [\langle s_i, h_0 \rangle, \langle s_i, h_1 \rangle, ..., \langle s_i, h_n \rangle] = [s_ih_0^\top, s_ih_1^\top, ..., s_ih_n^\top]$$

Теперь посмотрим на вероятностное распределение или это ещё называют `attention distribution`:

$$\alpha_i = softmax(e_i)$$

Теперь посчитаем взвешенную сумму (вес - это $\alpha_i$) для нахождения окончательного `attention vector`:

$$a_i = \sum_{k=0}^n \alpha_k h_k$$


Теперь в декодере на $i$-ом шаге вместо вектора `hidden state` $(h_0, h_1, ..., h_n)$ будем использовать `attention vector` $[s_i, a_i]$. Таким образом, на каждом шаге декодер получает информацию о важности всех токенов входной последовательности. 

Если в классической архитектуре `encoder-decoder` было так:

![alt](../data/enc-dec.png)

То теперь с появлением `attention` стало так:

![alt](../data/enc-dec-att.png)


В поведении токенов это можно представить себе так:

![alt](../data/attention.gif)


Например, слово "замок" может быть переведено как "castle" или как "lock". Для разрешения неоднозначности необходимо анализировать соседние слова:

1. Замок имеет шесть башен и окружён глубоким рвом -> castle.

2. Замок заржавел, и ключ в нём не поворачивается -> lock.

В первом случае внимание восстановит смысл по таким словам, как "башня" и "ров". А во втором - по слову "ключ".

Технически механизм внимания использует три сущности:

* Запрос (Queries)
* Ключи (Keys)
* Значения (Values)

Его работа вдохновлена SQL-запросами к базе данных вида

```sql
select VALUE where KEY=QUERY
```

в которых считывается значение записи (`VALUE`), у которой ключ (`KEY`) совпадает с запросом (`QUERY`).

### Self-attention

Несколько лет внимание работало только между разными последовательностями (энкодер и декодер). Но потом придумали гениально простую вещь:

`Self-Attention` (самовнимание) - процедуру, имплементируя которую, мы добиваемся эффекта того, что все токены внутри **одной и той же последовательности** смотря друг на друга.

> «Он взял свою книгу, потому что она была его любимой.»

Кто такой «он»? Какая книга «её»? `Self-attention` позволяет слову «она» «увидеть» слово «книга» несколькими шагами ранее.

По-сути, для последовательности из $n$ токенов мы имеем:

1. Токен с номером $i$ порождает три вектора:
    * $q_i = W_qx_i$
    * $k_i = W_kx_i$
    * $v_i = W_vx_i$

2. Считаем величину внимания `attention score` между токеном $i$ и всеми токенами $j$:
$$ e_{ij} = \langle q_i, k_j \rangle$$

3. Получаем распределение:

$$\alpha_{ij} = softmax_j(e_{ij})$$

4. Выход для токена $i$:

$$a_i = \sum_j \alpha_{ij}v_j$$


![alt](../data/lstm_vs_attention.png)

Эти идеи легли в создание современной архитектуры, про которую мы сейчас поговорим подробнее.

## Transformers

Механизм `self-attention` лёг в основу архитектуры `transformer`. Эта архитектура полностью отказалась от рекуррентности `RNN` и свёрток `CNN`, теперь используется только механизм внимания.

Предложена в статье [Attention is All You Need](https://arxiv.org/abs/1706.03762) (Vaswani et al., 2017). Название говорит само за себя: «Вам нужно только внимание».


Ниже приведено устройство архитектуры «трансформер» из оригинальной статьи

![alt](../data/transformer.svg)

Давайте теперь разберемся с каждым элементном этой штуки отдельно

### PE

![alt](../data/PE.png)

В отличие от `RNN` в `transformer` не закладывается по умолчанию информация о порядке токенов. Однако, если мы добавим позиционное кодирование, то всё будет гораздо лучше. Такой подход получил название `positional embeddings`. По сути, это некоторая добавка к "обычным" `embeddings` токенов.

Цель такого подхода заключается в том, чтобы подсказать языковой модели, что одно слово находится левее (или правее), чем другое слово.

$$PE_{(pos, 2i)} = sin(pos/10000^{\frac{2i}{d_{model}}})$$
$$PE_{(pos, 2i+1)} = cos(pos/10000^{\frac{2i}{d_{model}}})$$

* $pos$ - позиция в последовательности
* $i$ - индекс компоненты внутри `embedding`
* $d_{model}$ - размерность `embedding`

![alt](../data/PE_2.png)





### Multi-head attention (MHA)

![alt](../data/MHA.png)

Теперь обсудим основное тело `transformer`

![alt](../data/MHA2.png)

Эта формула очень похоже на то, что мы смотрели в прошлом разделе. Только здесь добавлены ещё два важных слоя:

1. `Scale` - слой нормировки. Просто поделим результат перемножения $Q$ и $K$ на корень из размерности `hidden state` для нормировки дисперсии. 

2. `Mask` - это слой маскирования. Во время обучения языковой модели, нам бы не хотелось смотреть на будущие токены (учитываем тот факт, что вся последовательность нам известна). Тогда для этого просто занулим все токены, которые стоят правее текущего.

Вот более точная иллюстрация того, что происходит в `MHA`


![alt](../data/MHA3.png)

Кстати, если мы переводим с одного языка на другой, то синтаксический порядок отдельных токенов может различаться в исходном и конечном языках. Это можно увидеть по матрице весов внимания:

![alt](../data/matrix.png)


### Feed-forward (FFN)

![alt](../data/FFN.png)

Теперь рассмотрим устройство следующего слоя `Feed-forward` (ещё это частно называют `MLP` или `FCNN`).
Этот блок состоит из двух матриц. Одна обеспечивает расширение размерности исходного, а вторая матрица обеспечивает обратное сжатие. Можно сказать, что `FFN` выступает неким хранилищем знаний модели (но так ли это.. непонятно..)

![alt](../data/FFN2.png)

$$FFN = RELU(xW_1 + b_1)W_2 + b_2$$

`Attention` находит связи между словами. `FFN` "думает" над каждым словом отдельно, применяя нелинейные преобразования. Без `FFN` `Transformer` был бы просто линейной комбинацией входов.

### LayerNorm

![alt](../data/ln.png)

Остался ещё один блок, который мы не рассмотрели. У этого слоя одна задача: перенормировать вектор после `MHA` или `FFN`. Это плохо для оптимизации (обучения).

$$LayerNorm(x) = \gamma \frac{x-\mu}{\sigma} + \beta$$

* Параметры $\gamma$ и $\beta$ являются настраиваемыми параметрами.

### LM-head

![alt](../data/LM_head.png)

1. Последним слоем в такой архитектуре будет линеный слой. Этот слой отображает вектор, получаемый из трансформера в вектор логитов.
2. Вектор логитов попадает в $softmax$ для получения вероятностей токенов.

![alt](../data/softmax.png)


Подход к обработке последовательностей целиком через внимание позволяет избавиться от такого понятия, как скрытое состояние, обновляющееся рекуррентно: каждый токен может напрямую «прочитать» любую часть последовательности, наиболее полезную для предсказания. В частности, отсутствие рекуррентности означает, что мы можем применять слой ко всей последовательности одновременно, так как матричные умножения прекрасно параллелятся.

Однако стоит помнить о затратах памяти и времени: поскольку каждый элемент последовательности взаимодействует с каждым, легко показать, что сложность self-attention составляет $O(n^2)$ по длине последовательности

Трансформер - это основа современного NLP!

## BERT и GPT

![alt](../data/GPT_BERT.png)

Несомненно, трансформер-модели не были бы так интересны, если бы практически все задачи NLP сейчас не решались бы с помощью этой архитектуры. Главными факторами, повлиявшими на бурный рост популярности идеи self-attention, послужили два семейства хорошо всем известных архитектур — BERT и GPT, которые в некотором роде являются энкодером и декодером трансформера, которые зажили своей жизнью.


### GPT

Модель `GPT` (**Generative Pretrained Transformer**) хронологически появилась раньше. Она представляет собой обычную языковую модель, реализованную в виде последовательности слоев декодера трансформера.

В качестве задачи при обучении выступает обычное предсказание следующего токена (то есть многоклассовая классификация по словарю). Важно, что в качестве маски внимания как раз выступает нижнетреугольная матрица: в противном случае возникла бы утечка в данных из-за того, что токены из «прошлого» будут видеть «будущее». Полученную модель можно использовать для генерации текстов и всех задач, которые на это опираются. Даже `ChatGPT`, обученная на специальных инструкциях, по своей сути незначительно отличается от базовой модели.

![alt](../data/GPT.svg)

Как понятно из названия, модель **Bidirectional Encoder Representations from Transformers** (или `BERT`) отличается от `GPT` двунаправленностью внимания: это значит, что при обработке входной последовательности все токены могут использовать информацию друг о друге.

Это делает такую архитектуру более удобной для задач, где нужно сделать предсказание относительно всего входа целиком без генерации, например, при классификации предложений или поиске пар похожих документов. Важно, что при этом `BERT` не учится генерировать тексты с нуля: одна из его задач при обучении — это `masked language modeling` (предсказание случайно замаскированных слов по оставшимся, изображено на рисунке ниже), а вторая — `next sentence prediction` (предсказание по паре текстовых фрагментов, следуют они друг за другом или нет).

![alt](../data/BERT.webp)


Заметим, что самое ключевое отличие в моделях `BERT` и `GPT` (а не в задачах для обучения или применениях) можно свести к использованию разных видов внимания.


<video width="50%" controls>
  <source src="../data/final.mp4" type="video/mp4">
  Ваш браузер не поддерживает видео.
</video>